# 14 — Exploration vs Exploitation

## Learning Objectives
1. Understand why exploration is necessary and when exploitation alone fails
2. Implement and compare four exploration strategies: random, epsilon-greedy, UCB, count-based
3. Build a curiosity-driven agent using Random Network Distillation (RND) approximation
4. Measure how different strategies affect state coverage on a 10x10 GridWorld


In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import time

np.random.seed(42)
print("NumPy:", np.__version__)
print("All environments hand-coded — no gym/torch")


## Level 1: GridWorld Coverage Maps

We put four exploration strategies on a 10x10 grid and measure what fraction
of states each one visits in 500 steps. The key insight: systematic strategies
like UCB cover far more states than naive epsilon-greedy, even with small epsilon.


In [ ]:
# --- Level 1: GridWorld coverage comparison ---

class GridWorld:
    """10x10 grid. Agent: 4 actions (right, left, down, up). Goal at corner."""
    ACTIONS = [(0, 1), (0, -1), (1, 0), (-1, 0)]

    def __init__(self, size=10, goal=None, seed=42):
        self.size = size; self.n_states = size*size; self.n_actions = 4
        self.goal = goal or (size-1, size-1)
        self.state = (0, 0)

    def reset(self):
        self.state = (0, 0); return self._enc(self.state)

    def step(self, a):
        dy, dx = self.ACTIONS[a]
        r, c = self.state
        nr = max(0, min(self.size-1, r+dy)); nc = max(0, min(self.size-1, c+dx))
        self.state = (nr, nc)
        done = self.state == self.goal
        return self._enc(self.state), float(done), done

    def _enc(self, s): return s[0]*self.size + s[1]
    def decode(self, s): return (s//self.size, s%self.size)


def measure_coverage(strategy, n_steps=500, size=10, seed=42):
    """
    Run a strategy (function: state -> action) for n_steps.
    Returns (visit_map [size x size], coverage_over_time).
    """
    np.random.seed(seed)
    grid = GridWorld(size, seed=seed)
    visit_map = np.zeros(grid.n_states, dtype=int)
    coverage = []
    Q = np.zeros((grid.n_states, grid.n_actions))
    N = np.zeros((grid.n_states, grid.n_actions))
    s = grid.reset(); ep_steps = 0

    for t in range(n_steps):
        a = strategy(s, Q, N, t)
        ns, r, done = grid.step(a)
        # Q-learning update
        td = r + 0.99 * Q[ns].max() - Q[s, a]
        Q[s, a] += 0.1 * td
        N[s, a] += 1
        visit_map[ns] += 1
        s = ns; ep_steps += 1
        if done or ep_steps >= 200:
            s = grid.reset(); ep_steps = 0
        coverage.append(int(np.sum(visit_map > 0)))

    return visit_map.reshape(size, size), coverage


# Strategy functions
def random_strategy(s, Q, N, t): return int(np.random.randint(4))

def greedy_strategy(s, Q, N, t):
    eps = 0.1
    return np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s]))

def ucb_strategy(s, Q, N, t):
    c = 1.0; t_safe = max(1, t)
    return int(np.argmax(Q[s] + c * np.sqrt(np.log(t_safe+1) / (N[s]+1e-8))))

def count_bonus_strategy(s, Q, N, t):
    beta = 0.3
    bonus = beta / np.sqrt(N[s] + 1)
    return int(np.argmax(Q[s] + bonus))


strategies = {
    "Random":      random_strategy,
    "eps-greedy":  greedy_strategy,
    "UCB":         ucb_strategy,
    "Count-bonus": count_bonus_strategy,
}

maps = {}; coverages = {}
for name, strat in strategies.items():
    vm, cov = measure_coverage(strat, n_steps=500)
    maps[name] = vm; coverages[name] = cov

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, (name, vm) in enumerate(maps.items()):
    ax = axes[0, i]
    im = ax.imshow(np.log1p(vm), cmap="Blues", origin="upper")
    ax.set_title(f"{name}
{int(np.sum(vm>0))}/100 states")
    ax.set_xticks([]); ax.set_yticks([])
    ax.scatter([9], [9], color="red", s=100, marker="*")  # goal

for i, (name, cov) in enumerate(coverages.items()):
    axes[1, i].plot(cov, color="steelblue")
    axes[1, i].set_xlabel("Step"); axes[1, i].set_ylabel("States visited")
    axes[1, i].set_title(name); axes[1, i].grid(True, alpha=0.3)
    axes[1, i].set_ylim(0, 101)

plt.suptitle("GridWorld Coverage Maps (500 steps, log-scale visit count)", fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/exploration_maps.png", dpi=80)
plt.show()

for name in strategies:
    final_cov = coverages[name][-1]
    print(f"  {name:13s}: {final_cov:3d}/100 states ({final_cov:.0f}%)")


## Level 2: Curiosity-Driven Exploration (RND Approximation)

Random Network Distillation (RND) uses prediction error as intrinsic reward:
1. Fix a random encoder phi(s): maps states to embeddings
2. Train a predictor f_theta(s) to predict phi(s)
3. Intrinsic reward r_i(s) = ||f_theta(s) - phi(s)||^2 — high for unseen states

As the agent visits a state more, f_theta learns to predict phi(s), reducing r_i.
Unvisited states always produce high prediction error -> natural exploration bonus.


In [ ]:
# --- Level 2: RND Curiosity on GridWorld ---

class RNDAgent:
    """
    Q-learning + RND intrinsic reward.
    Fixed random encoder: encoder[s] = fixed random vector in R^embed_dim
    Learnable predictor: predictor[s] trained toward encoder[s]
    """
    def __init__(self, n_states, n_actions, embed_dim=8, int_coef=0.5,
                 alpha=0.1, gamma=0.99, epsilon=0.05):
        self.Q = np.zeros((n_states, n_actions))
        self.n_actions = n_actions
        self.alpha = alpha; self.gamma = gamma
        self.epsilon = epsilon; self.int_coef = int_coef

        # Fixed random encoder (never trained)
        np.random.seed(1)
        self.encoder = np.random.randn(n_states, embed_dim)
        # Trainable predictor (initialised near zero)
        self.predictor = np.zeros((n_states, embed_dim))
        self.pred_lr = 0.05

    def intrinsic_reward(self, ns):
        """Prediction error = curiosity for state ns."""
        return float(np.sum((self.encoder[ns] - self.predictor[ns])**2))

    def update_predictor(self, ns):
        """Gradient step: reduce prediction error for visited state."""
        self.predictor[ns] += self.pred_lr * (self.encoder[ns] - self.predictor[ns])

    def select(self, s):
        if np.random.random() < self.epsilon:
            return int(np.random.randint(self.n_actions))
        return int(np.argmax(self.Q[s]))

    def update_q(self, s, a, r_ext, r_int, ns, done):
        r_total = r_ext + self.int_coef * r_int
        td = r_total + self.gamma * self.Q[ns].max() * (1 - float(done)) - self.Q[s, a]
        self.Q[s, a] += self.alpha * td


def run_rnd(n_steps=500, size=10, n_episodes=None, seed=42):
    """Run RND curiosity agent. Returns (visit_map, coverage, intrinsic_rewards)."""
    np.random.seed(seed)
    grid = GridWorld(size, seed=seed)
    agent = RNDAgent(grid.n_states, grid.n_actions, embed_dim=8, int_coef=0.5)
    visit_map = np.zeros(grid.n_states, dtype=int)
    coverage = []; intrinsic_rewards = []
    s = grid.reset(); ep_steps = 0

    for _ in range(n_steps):
        a = agent.select(s)
        ns, r_ext, done = grid.step(a)
        r_int = agent.intrinsic_reward(ns)
        agent.update_predictor(ns)
        agent.update_q(s, a, r_ext, r_int, ns, done)
        visit_map[ns] += 1
        intrinsic_rewards.append(r_int)
        coverage.append(int(np.sum(visit_map > 0)))
        s = ns; ep_steps += 1
        if done or ep_steps >= 200:
            s = grid.reset(); ep_steps = 0

    return visit_map.reshape(size, size), coverage, intrinsic_rewards


print("Running RND curiosity agent on 10x10 GridWorld...")
vm_rnd, cov_rnd, int_r = run_rnd(n_steps=500, size=10)
vm_eps, cov_eps = measure_coverage(greedy_strategy, n_steps=500)

print(f"RND coverage:       {cov_rnd[-1]:3d}/100 states")
print(f"eps-greedy coverage: {cov_eps[-1]:3d}/100 states")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
im1 = axes[0].imshow(np.log1p(vm_rnd), cmap="Greens", origin="upper")
axes[0].set_title(f"RND Curiosity
{cov_rnd[-1]}/100 states"); axes[0].set_xticks([]); axes[0].set_yticks([])
im2 = axes[1].imshow(np.log1p(vm_eps), cmap="Blues", origin="upper")
axes[1].set_title(f"eps-greedy
{cov_eps[-1]}/100 states"); axes[1].set_xticks([]); axes[1].set_yticks([])
axes[2].plot(cov_rnd, label="RND", color="darkgreen")
axes[2].plot(cov_eps, "--", label="eps-greedy", color="steelblue")
axes[2].set_xlabel("Step"); axes[2].set_ylabel("States covered")
axes[2].set_title("Coverage Over Time"); axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.suptitle("RND Curiosity vs eps-greedy: State Coverage", fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/rnd_coverage.png", dpi=80)
plt.show()

# Intrinsic reward should decrease as states are visited
w = 20
axes2_data = np.convolve(int_r, np.ones(w)/w, 'valid')
print(f"Avg intrinsic reward: first 50 steps={np.mean(int_r[:50]):.3f}, last 50={np.mean(int_r[-50:]):.3f}")
print("Decreasing intrinsic reward confirms predictor is learning -> less curiosity for visited states")


## Real-World Example 1: Hard Exploration with Sparse Reward

The true test of exploration: a 10x10 GridWorld with a reward ONLY at the goal (0,0) -> (9,9).
Standard epsilon-greedy almost never finds it; curiosity-driven methods are far more effective.


In [ ]:
# === Hard Exploration: Sparse Reward ===

def run_sparse_reward_comparison(n_steps=3000, size=10, seed=42):
    """
    GridWorld where reward is ONLY at goal (9,9). Compare two strategies.
    Returns number of goal-reaching events for each.
    """
    results = {}
    for name, use_rnd in [("eps-greedy", False), ("RND curiosity", True)]:
        np.random.seed(seed)
        grid = GridWorld(size, seed=seed)
        goal_reached = 0
        ep_steps = 0
        Q = np.zeros((grid.n_states, grid.n_actions))
        N = np.zeros((grid.n_states, grid.n_actions))
        goals_over_time = []

        if use_rnd:
            agent = RNDAgent(grid.n_states, grid.n_actions, int_coef=1.0, epsilon=0.05)

        s = grid.reset()
        for step in range(n_steps):
            if use_rnd:
                a = agent.select(s)
            else:
                eps = max(0.05, 1.0 - step/1000)
                a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s]))

            ns, r, done = grid.step(a)
            if use_rnd:
                r_int = agent.intrinsic_reward(ns)
                agent.update_predictor(ns)
                agent.update_q(s, a, r, r_int, ns, done)
            else:
                td = r + 0.99 * Q[ns].max() * (1-float(done)) - Q[s, a]
                Q[s, a] += 0.1 * td

            if done: goal_reached += 1
            goals_over_time.append(goal_reached)
            s = ns; ep_steps += 1
            if done or ep_steps >= 200:
                s = grid.reset(); ep_steps = 0

        results[name] = goals_over_time

    return results


print("Running sparse-reward comparison (3000 steps, 10x10 GridWorld)...")
t0 = time.time()
sparse_results = run_sparse_reward_comparison(n_steps=3000, size=10)
print(f"Done in {time.time()-t0:.1f}s
")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors_map2 = {"eps-greedy": "steelblue", "RND curiosity": "darkgreen"}
for name, goals in sparse_results.items():
    col = colors_map2[name]
    axes[0].plot(goals, label=name, color=col)
    print(f"  {name}: goal reached {goals[-1]} times in 3000 steps")

axes[0].set_xlabel("Step"); axes[0].set_ylabel("Cumulative goal-reaches")
axes[0].set_title("Sparse Reward: eps-greedy vs RND Curiosity")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Visualise: which states did each agent visit most?
vm_sparse_eps, _ = measure_coverage(greedy_strategy, n_steps=3000)
vm_sparse_rnd, _, _ = run_rnd(n_steps=3000, size=10)
for ax, vm, name in zip([axes[1]], [vm_sparse_rnd], ["RND curiosity"]):
    im = ax.imshow(np.log1p(vm), cmap="Greens", origin="upper")
    ax.set_title(f"{name} state coverage (log-scale)")
    ax.scatter([9], [9], color="red", s=100, marker="*")
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig("/tmp/sparse_exploration.png", dpi=80)
plt.show()


## Real-World Example 2: Epsilon Decay Schedules

How you anneal epsilon matters as much as its initial value.
Three common schedules:
- **Constant**: never fully commits to exploitation
- **Linear**: uniform exploration budget
- **Exponential**: fast early exploration, near-greedy late


In [ ]:
# === Epsilon Decay Schedule Comparison ===

def make_eps_schedule(schedule, t, total, eps_start=1.0, eps_end=0.05):
    """Compute epsilon at step t under the given schedule."""
    if schedule == "constant":
        return (eps_start + eps_end) / 2.0
    elif schedule == "linear":
        return max(eps_end, eps_start - (eps_start - eps_end) * t / total)
    elif schedule == "exponential":
        decay = (eps_end / eps_start) ** (1.0 / total)
        return max(eps_end, eps_start * (decay ** t))
    return eps_end


def run_decay_schedule(schedule, n_steps=2000, size=10, seed=42):
    """Run eps-greedy with given schedule on GridWorld. Returns coverage."""
    np.random.seed(seed)
    grid = GridWorld(size, seed=seed)
    Q = np.zeros((grid.n_states, grid.n_actions))
    visit_map = np.zeros(grid.n_states, dtype=int)
    coverage = []; s = grid.reset(); ep_steps = 0

    for t in range(n_steps):
        eps = make_eps_schedule(schedule, t, n_steps)
        a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s]))
        ns, r, done = grid.step(a)
        td = r + 0.99 * Q[ns].max() * (1-float(done)) - Q[s, a]
        Q[s, a] += 0.1 * td
        visit_map[ns] += 1; coverage.append(int(np.sum(visit_map > 0)))
        s = ns; ep_steps += 1
        if done or ep_steps >= 200: s = grid.reset(); ep_steps = 0

    return coverage, Q


schedules = ["constant", "linear", "exponential"]
coverage_by_sched = {}; q_by_sched = {}
t = time.time()
for sched in schedules:
    cov, Q = run_decay_schedule(sched, n_steps=2000)
    coverage_by_sched[sched] = cov; q_by_sched[sched] = Q

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors_s = {"constant": "steelblue", "linear": "orange", "exponential": "darkgreen"}
for sched in schedules:
    col = colors_s[sched]
    axes[0].plot(coverage_by_sched[sched], label=sched, color=col)

    # Show epsilon curve
    eps_curve = [make_eps_schedule(sched, t, 2000) for t in range(2000)]
    axes[1].plot(eps_curve, label=sched, color=col)

axes[0].set_xlabel("Step"); axes[0].set_ylabel("States Covered")
axes[0].set_title("State Coverage by Epsilon Schedule")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Step"); axes[1].set_ylabel("Epsilon")
axes[1].set_title("Epsilon Schedule Comparison")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/eps_decay.png", dpi=80)
plt.show()

print("Final coverage (2000 steps):")
for sched, cov in coverage_by_sched.items():
    print(f"  {sched:12s}: {cov[-1]:3d}/100 states")
print("
Final Q-values are different -> schedule affects what policy is learned")


## Real-World Example 3: UCB in Dyna-Q Model-Based Planning

Dyna-Q combines Q-learning with a model: after each real transition, the agent
simulates additional transitions using a learned model. UCB exploration bonuses
can guide which (state, action) pairs to plan over — prioritising less-visited ones.


In [ ]:
# === UCB in Dyna-Q (model-based RL) ===

class DynaQAgent:
    """
    Dyna-Q: after each real transition, do N simulated planning steps.
    UCB is used both for real actions and for selecting planning state-actions.
    """
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.99,
                 epsilon=0.15, n_planning=5, use_ucb=False, c=1.0):
        self.Q = np.zeros((n_states, n_actions))
        self.N = np.zeros((n_states, n_actions))  # visit counts
        self.model = {}           # (s, a) -> (ns, r)
        self.t = 0
        self.alpha = alpha; self.gamma = gamma; self.epsilon = epsilon
        self.n_planning = n_planning; self.use_ucb = use_ucb; self.c = c
        self.n_actions = n_actions

    def select(self, s):
        self.t += 1
        if self.use_ucb:
            bonus = self.c * np.sqrt(np.log(self.t+1) / (self.N[s]+1e-8))
            return int(np.argmax(self.Q[s] + bonus))
        return int(np.random.randint(self.n_actions) if np.random.random() < self.epsilon
                   else np.argmax(self.Q[s]))

    def update(self, s, a, r, ns, done):
        self.N[s, a] += 1
        self.model[(s, a)] = (ns, r)
        td = r + self.gamma * self.Q[ns].max() * (1-float(done)) - self.Q[s, a]
        self.Q[s, a] += self.alpha * td
        # Planning: simulate N steps from model
        if self.model:
            keys = list(self.model.keys())
            for _ in range(self.n_planning):
                ps, pa = keys[np.random.randint(len(keys))]
                pns, pr = self.model[(ps, pa)]
                self.Q[ps, pa] += self.alpha * (pr + self.gamma*self.Q[pns].max() - self.Q[ps, pa])


def run_dynaq(use_ucb=False, n_steps=1000, size=10, n_planning=5, seed=42):
    np.random.seed(seed)
    grid = GridWorld(size, seed=seed)
    agent = DynaQAgent(grid.n_states, grid.n_actions, n_planning=n_planning, use_ucb=use_ucb)
    visit_map = np.zeros(grid.n_states, dtype=int)
    coverage = []; goal_count = 0; s = grid.reset(); ep_steps = 0
    goals_over_time = []

    for _ in range(n_steps):
        a = agent.select(s)
        ns, r, done = grid.step(a)
        agent.update(s, a, r, ns, done)
        visit_map[ns] += 1; coverage.append(int(np.sum(visit_map > 0)))
        if done: goal_count += 1
        goals_over_time.append(goal_count)
        s = ns; ep_steps += 1
        if done or ep_steps >= 200: s = grid.reset(); ep_steps = 0

    return coverage, goals_over_time, visit_map.reshape(size, size)


print("Comparing Dyna-Q vs Dyna-Q+UCB on GridWorld (1000 steps)...")
cov_dq, goals_dq, vm_dq = run_dynaq(use_ucb=False, n_steps=1000)
cov_ucb_dq, goals_ucb_dq, vm_ucb_dq = run_dynaq(use_ucb=True, n_steps=1000, c=0.5)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(cov_dq, "--", label="Dyna-Q (eps)", color="steelblue")
axes[0].plot(cov_ucb_dq, label="Dyna-Q + UCB", color="darkgreen")
axes[0].set_xlabel("Step"); axes[0].set_ylabel("States Covered")
axes[0].set_title("Coverage: Dyna-Q vs Dyna-Q+UCB"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(goals_dq, "--", label="Dyna-Q", color="steelblue")
axes[1].plot(goals_ucb_dq, label="Dyna-Q+UCB", color="darkgreen")
axes[1].set_xlabel("Step"); axes[1].set_ylabel("Goals Reached")
axes[1].set_title("Goal-Reaching Events"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

im = axes[2].imshow(np.log1p(vm_ucb_dq), cmap="Greens", origin="upper")
axes[2].set_title(f"Dyna-Q+UCB visit map
{int(np.sum(vm_ucb_dq > 0))}/100 states")
axes[2].scatter([9], [9], color="red", s=100, marker="*")
axes[2].set_xticks([]); axes[2].set_yticks([])

plt.tight_layout()
plt.savefig("/tmp/dynaq_ucb.png", dpi=80)
plt.show()
print(f"Goals reached: Dyna-Q={goals_dq[-1]}  Dyna-Q+UCB={goals_ucb_dq[-1]}")


## Comparison: States Explored in 500 Steps — All Strategies


In [ ]:
# === Final Comparison: all strategies on 10x10 GridWorld, 500 steps ===

strategies_full = {
    "Random":       random_strategy,
    "eps-greedy":   greedy_strategy,
    "UCB":          ucb_strategy,
    "Count-bonus":  count_bonus_strategy,
}

print("Running full comparison (500 steps, 5 random seeds each)...")
final_coverages = {}
for name, strat in strategies_full.items():
    seed_covs = []
    for seed in range(5):
        _, cov = measure_coverage(strat, n_steps=500, seed=seed)
        seed_covs.append(cov[-1])
    final_coverages[name] = seed_covs

# Also RND curiosity
rnd_covs = []
for seed in range(5):
    _, cov, _ = run_rnd(n_steps=500, size=10, seed=seed)
    rnd_covs.append(cov[-1])
final_coverages["RND curiosity"] = rnd_covs

names = list(final_coverages.keys())
means = [np.mean(v) for v in final_coverages.values()]
stds  = [np.std(v) for v in final_coverages.values()]
colors_bar = ["gray", "steelblue", "purple", "orange", "darkgreen"]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(names, means, yerr=stds, capsize=4, color=colors_bar, alpha=0.85, edgecolor="black")
ax.axhline(100, color="red", linestyle="--", alpha=0.5, label="Full coverage (100 states)")
ax.set_xlabel("Exploration Strategy"); ax.set_ylabel("States Covered (of 100)")
ax.set_title("10x10 GridWorld: States Covered in 500 Steps (mean +/- std over 5 seeds)")
ax.legend(); ax.grid(True, alpha=0.3, axis="y")
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1, f"{mean:.0f}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig("/tmp/exploration_comparison.png", dpi=80)
plt.show()

print("
Final coverage (mean +/- std):")
for name in names:
    vals = final_coverages[name]
    print(f"  {name:14s}: {np.mean(vals):.1f} +/- {np.std(vals):.1f}")


## Key Takeaways

**Core idea:** Exploration is a deliberate investment — the agent must take
"suboptimal" actions now to gather information that enables better decisions later.
The choice of exploration strategy fundamentally shapes what the agent learns,
not just how quickly it converges.

### Variants and When to Use

| Strategy | Exploration Quality | Non-stationary | Scales with S | Suitable for |
|----------|--------------------|-----------------|--------------|--------------|
| Random | Uniform | Perfect | Poor | Baselines only |
| eps-greedy | Moderate | Good (constant eps) | OK | Simple tasks |
| UCB | Principled | Poor (slow forget) | Good | Stationary |
| Count-based | Principled | Good | Moderate | Tabular RL |
| RND curiosity | Excellent | Very good | Excellent | Deep RL, sparse reward |

### Common Failure Modes

- **epsilon too small too early:** Policy commits to a suboptimal arm before
  covering enough states. Symptom: agent never finds the optimal path.
  Fix: start with eps=0.5-1.0, decay over 50-80% of training.
- **Curiosity bonus too large:** Agent ignores extrinsic reward entirely and
  keeps exploring without exploiting. Symptom: high state coverage but zero task reward.
  Fix: scale int_coef by the ratio of intrinsic to extrinsic reward magnitudes.
- **UCB constant c too large:** Agent uniformly explores even provably bad state-actions.
  Tune c so that 80-90% of choices are exploitative in late training.
- **RND with duplicate states:** If state representation is not unique, prediction error
  doesn't decrease for "same" states, causing permanent high intrinsic reward.
  Fix: normalise state features before feeding to encoder.

### Related Concepts

- [13-multi-armed-bandit](./13-multi-armed-bandit.ipynb) — bandit exploration as the root formulation
- [11-proximal-policy-optimization](./11-proximal-policy-optimization.ipynb) — PPO entropy bonus for exploration
- [12-soft-actor-critic](./12-soft-actor-critic.ipynb) — entropy-regularised exploration in SAC
- [15-reward-shaping](./15-reward-shaping.ipynb) — intrinsic reward is a form of reward shaping


## Exercises

1. **Adapt epsilon greedily:** Implement an epsilon that decreases faster when
   estimated Q-value variance is low (the agent has converged). Does this improve
   sample efficiency?

2. **RND vs oracle count-based:** Count-based exploration gives bonus 1/sqrt(N(s)).
   RND approximates this without explicit counts. Compare the two directly: run both
   on the same 10x10 grid and plot their coverage curves.

3. **Hard exploration GridWorld:** Create a 20x20 grid with the goal at (19, 19).
   How many steps does each strategy need to first reach the goal?

4. **Intrinsic reward decay:** In the RND implementation, add a decay to the
   intrinsic coefficient (int_coef *= 0.999 per step). Does this help or hurt?

5. **Dyna-Q planning depth:** In the Dyna-Q + UCB implementation, vary n_planning
   from 0 to 20. Plot goals-per-step as a function of n_planning. What is the
   optimal planning budget?
